In [ ]:
# %pip install langchain
# %pip install langchain-community
# %pip install langchain-neo4j       # Official Neo4j LangChain integration
# %pip install langchain-openai
# %pip install langchain-experimental  # LLMGraphTransformer
# %pip install langgraph             # Agentic workflow engine

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load from .env file

# --- API Keys ---
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "sk-your-key-here")

# --- Neo4j Connection ---
# Option A: Local Neo4j Desktop / Docker
os.environ["NEO4J_URI"]      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
os.environ["NEO4J_USERNAME"] = os.getenv("NEO4J_USERNAME", "neo4j")
os.environ["NEO4J_PASSWORD"] = os.getenv("NEO4J_PASSWORD", "password")

In [2]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    enhanced_schema=True,   # Richer schema for Cypher generation
    sanitize=True,          # Clean unsafe characters
    refresh_schema=True,    # Always fetch latest schema
)

In [3]:
from langchain_openai import OpenAIEmbeddings,ChatOpenAI

llm=ChatOpenAI(model_name="gpt-4o")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x13503b8f0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1383e4740>, root_client=<openai.OpenAI object at 0x10ad3c740>, root_async_client=<openai.AsyncOpenAI object at 0x13503b950>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [4]:
from langchain_core.documents import Document
text="""
Elon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space
company SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp.,
formerly Twitter, and his role in the founding of The Boring Company, xAI, Neuralink and OpenAI.
He is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be
US$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended
the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through
his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.
Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics
 and physics. He moved to California in 1995 to attend Stanford University, but dropped out after
  two days and, with his brother Kimbal, co-founded online city guide software company Zip2.
 """
documents=[Document(page_content=text)]
documents

[Document(metadata={}, page_content="\nElon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space\ncompany SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp.,\nformerly Twitter, and his role in the founding of The Boring Company, xAI, Neuralink and OpenAI.\nHe is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be\nUS$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended\nthe University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through\nhis Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.\nMusk later transferred to the University of Pennsylvania and received bachelor's degrees in economics\n and physics. He moved to California in 1995 to attend Stanford University, but dropped out after\n  two days and, with his brother Kimbal, co-founded onlin

In [5]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer=LLMGraphTransformer(llm=llm)

In [6]:
graph_documents=llm_transformer.convert_to_graph_documents(documents)

/Users/sourangshupal/Downloads/graph-rag/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DynamicGraph(nodes=[Simpl...n', type='co-founded')]), input_type=DynamicGraph])
  return self.__pydantic_serializer__.to_python(


In [7]:
graph_documents

[GraphDocument(nodes=[Node(id='Elon Reeve Musk', type='Person', properties={}), Node(id='Spacex', type='Organization', properties={}), Node(id='Tesla, Inc.', type='Organization', properties={}), Node(id='X Corp.', type='Organization', properties={}), Node(id='The Boring Company', type='Organization', properties={}), Node(id='Xai', type='Organization', properties={}), Node(id='Neuralink', type='Organization', properties={}), Node(id='Openai', type='Organization', properties={}), Node(id='Forbes', type='Organization', properties={}), Node(id='July 2024', type='Date', properties={}), Node(id='Us$221 Billion', type='Amount', properties={}), Node(id='Pretoria', type='Place', properties={}), Node(id='Maye', type='Person', properties={}), Node(id='Errol Musk', type='Person', properties={}), Node(id='University Of Pretoria', type='Organization', properties={}), Node(id='Canada', type='Place', properties={}), Node(id="Queen'S University At Kingston", type='Organization', properties={}), Node(id

In [8]:
graph_documents[0].nodes

[Node(id='Elon Reeve Musk', type='Person', properties={}),
 Node(id='Spacex', type='Organization', properties={}),
 Node(id='Tesla, Inc.', type='Organization', properties={}),
 Node(id='X Corp.', type='Organization', properties={}),
 Node(id='The Boring Company', type='Organization', properties={}),
 Node(id='Xai', type='Organization', properties={}),
 Node(id='Neuralink', type='Organization', properties={}),
 Node(id='Openai', type='Organization', properties={}),
 Node(id='Forbes', type='Organization', properties={}),
 Node(id='July 2024', type='Date', properties={}),
 Node(id='Us$221 Billion', type='Amount', properties={}),
 Node(id='Pretoria', type='Place', properties={}),
 Node(id='Maye', type='Person', properties={}),
 Node(id='Errol Musk', type='Person', properties={}),
 Node(id='University Of Pretoria', type='Organization', properties={}),
 Node(id='Canada', type='Place', properties={}),
 Node(id="Queen'S University At Kingston", type='Organization', properties={}),
 Node(id='Un

In [9]:
graph_documents[0].relationships

[Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Spacex', type='Organization', properties={}), type='KEY_ROLE', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Tesla, Inc.', type='Organization', properties={}), type='KEY_ROLE', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='X Corp.', type='Organization', properties={}), type='OWNER', properties={}),
 Relationship(source=Node(id='X Corp.', type='Organization', properties={}), target=Node(id='Twitter', type='Organization', properties={}), type='FORMERLY_KNOWN_AS', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='The Boring Company', type='Organization', properties={}), type='FOUNDING_ROLE', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Xai', t

In [10]:
### Load the dataset of movie

movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In [11]:
graph

In [12]:
graph.query(movie_query)

[]

In [13]:
graph.refresh_schema()
print(graph.schema)

Node properties:
- **Movie**
  - `id`: STRING Example: "1"
  - `released`: DATE Min: 1964-12-16, Max: 1996-09-15
  - `title`: STRING Example: "Toy Story"
  - `imdbRating`: FLOAT Min: 2.4, Max: 9.3
- **Person**
  - `name`: STRING Example: "John Lasseter"
- **Genre**
  - `name`: STRING Example: "Adventure"
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [16]:
from langchain_neo4j import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(llm=llm,graph=graph,verbose=True, allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x12b5fe720>, cypher_generation_chain=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x13503b8f0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1383e4740>, root_client=<opena

In [17]:
response=chain.invoke({"query":"Who was the director of the moview GoldenEye"})

response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "GoldenEye"})
RETURN p.name

Full Context:
[{'p.name': 'Martin Campbell'}]

> Finished chain.


{'query': 'Who was the director of the moview GoldenEye',
 'result': 'Martin Campbell was the director of the movie GoldenEye.'}

In [18]:
response=chain.invoke({"query":"tell me the genre of the movie GoldenEye"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (m:Movie {title: 'GoldenEye'})-[:IN_GENRE]->(g:Genre)
RETURN g.name

Full Context:
[{'g.name': 'Adventure'}, {'g.name': 'Action'}, {'g.name': 'Thriller'}]

> Finished chain.


{'query': 'tell me the genre of the movie GoldenEye',
 'result': 'Adventure, Action, Thriller'}

In [19]:
response=chain.invoke({"query":"Who was the director in movie Casino"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name

Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


{'query': 'Who was the director in movie Casino',
 'result': 'Martin Scorsese was the director of the movie Casino.'}